# All approaches with vLLM (test or val, `semextract_revised`)
Runs every text input mode under both table settings, and every fusion paradigm on top of each, with the same context and labels:

| variant | what |
|---|---|
| `text_<mode>_<setting>` | text only; `in_text` (A) with HTML tables, `separate` (B) without table bodies |
| `tables` (B), `images` | once, shared by all variants |
| `text_<mode>_separate_with_tables` | B text + tables, rule-based late fusion (counterpart of `text_<mode>_in_text`) |
| `baseline` | once: unfiltered, all text, tables, figures and transcriptions |
| `early_<mode>_<setting>` | early fusion: text and relevant figures together |
| `<order>_<mode>_<setting>` | intermediate fusion, e.g. `text_then_image_filtered_in_text` |
| `late_rule_<mode>_<setting>`, `late_llm_<mode>_<setting>` | late fusion of the setting's modalities |

`<mode>` is `entire`, `filtered` or `elementwise`. Every variant is reviewed against its own sources; before/after versions are saved separately. Trim `TEXT_INPUT_MODES`, `INTERMEDIATE_ORDERS` or `LATE_PARADIGMS` to run fewer.

Configure the first code cell after setup:
- `SPLIT`: `'test'` (`mineru/test_pdfs`) or `'val'` (`mineru/val_pdfs`).
- `LOAD_CONTEXTS` / `LOAD_LABELS`: load contexts and source labels from saved files instead of generating them with the model. Sources without a saved label are classified.
- `RESUME_STAGE_DIR`: `None` starts a new run; an earlier run's folder continues it. Completed variants are reused, the rest is run; the run's own contexts and labels are used.



In [ ]:
from pathlib import Path
import os
import sys

# Set this to your project directory, or set the SEMEXTRACT_PROJECT environment variable.
# None discovers the project from the current working directory or its parents.
PROJECT_OVERRIDE = os.environ.get("SEMEXTRACT_PROJECT")  # e.g. "/path/to/multimodal_extraction"

if PROJECT_OVERRIDE:
    PROJECT = Path(PROJECT_OVERRIDE).expanduser().resolve()
else:
    PROJECT = next(
        (p for p in (Path.cwd(), *Path.cwd().parents)
         if (p / "semextract_revised").is_dir()),
        None,
    )
if PROJECT is None or not (PROJECT / "semextract_revised").is_dir():
    raise FileNotFoundError("Set PROJECT_OVERRIDE to your multimodal_extraction directory.")
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))


def project_path(value):
    """Resolve an absolute path, or a path relative to PROJECT; expand ~."""
    path = Path(value).expanduser()
    return (path if path.is_absolute() else PROJECT / path).resolve()


print("Project directory:", PROJECT)

import time
import json
import copy
from collections import Counter

if not (PROJECT / "prompts_revise/manifest.json").is_file():
    raise FileNotFoundError(f"Prompt manifest not found in {PROJECT / 'prompts_revise'}")


In [ ]:
# Run once if the notebook environment does not already have the dependencies.
import subprocess
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT / 'requirements-vllm.txt')],
    check=True,
)


Restart the Python kernel if installation changed Torch/CUDA, then rerun setup. The revised pipeline does not render Graphviz review images, so no `dot` executable is needed.


In [ ]:
import semextract_revised.extraction as ex
from semextract_revised.common import prompt_fingerprint, read_json, write_json
from semextract_revised.documents_all_images import load_article
from semextract_revised.inputs import condition_for, default_modalities, load_contexts, normalized_labels
from semextract_revised.runtime import engine_settings, new_run_dir

SPLIT = 'test'  # 'test' or 'val'
if SPLIT not in ('test', 'val'):
    raise ValueError("SPLIT must be 'test' or 'val'.")

# Edit these directories, or set the environment variables. Absolute paths and
# paths relative to PROJECT are both accepted. DATA_DIR contains <article>/vlm/*_content_list.json.
DATA_DIR = project_path(os.environ.get('SEMEXTRACT_DATA_DIR', f'mineru/{SPLIT}_pdfs'))
OUTPUT_ROOT = project_path(os.environ.get('SEMEXTRACT_OUTPUT_ROOT', f'out/stages/vllm_{SPLIT}'))

# Saved inputs. What is not loaded is generated with the model in sections 1 and 2.
LOAD_CONTEXTS = True
CONTEXTS_PATH = project_path('out/stages/vllm_test/contexts.json')
LOAD_LABELS = True
LABELS_PATH = project_path('out/stages/vllm_test/labels.json')

# None starts a new run. To continue a run, enter its folder (printed below).
RESUME_STAGE_DIR = None
# RESUME_STAGE_DIR = project_path('out/stages/vllm_test')

ex.MODEL_NAME = 'Qwen/Qwen3.8-27B'  # choose your installed/supported checkpoint
ex.MAX_TOKENS = 32768            # output tokens per request
ex.CONTEXT_MAX_TOKENS = 32768    # general pass (default 16384)
ex.CUT_OFF_RETRY_FACTOR = 2
ex.MAX_ENGINE_IMAGES = 8  # baseline/early fusion: all relevant figures of a paper in one request (more are split)
ex.VISUAL_TOKEN_BUDGET = 65536  # visual tokens per request; per-image resolution shrinks with more images
ex.ENABLE_THINKING = False
ex.REVISE_TEXT = ex.REVISE_IMAGES = ex.REVISE_TABLES = ex.REVISE_JOINT = True
ex.TEXT_DECIMAL_COMMA = False
ex.FILTER_CONTENT = True

# Text input modes, each under both table settings.
#   entire      whole text, no classification filter
#   filtered    only sources classified relevant, one request per article
#   elementwise one request per relevant source
TEXT_INPUT_MODES = ['entire', 'filtered', 'elementwise']
# A 'in_text': HTML tables inside the text. B 'separate': text without table bodies + own table extraction.
TABLE_SETTINGS = ['in_text', 'separate']
TABLE_REPRESENTATION = 'html'  # B: 'html' or 'image' (a table lacking that form falls back to the other)
UPSTREAM_VERSION = 'after_revision'  # which single-modality version every fusion consumes

# Fusion paradigms: each runs once per table setting and text mode,
# reusing that variant's text result plus the shared images (and tables in B).
FUSION_MODALITIES = {s: default_modalities(s) for s in TABLE_SETTINGS}  # A: text+images, B: text+tables+images
INTERMEDIATE_ORDERS = {'in_text': ['image_then_text', 'text_then_image'],
                       'separate': ['text_then_tables_then_image', 'image_then_tables_then_text']}
LATE_PARADIGMS = ['late_rule', 'late_llm']
T = 0.1
ENGINE = engine_settings(ex.MODEL_NAME, ex.MAX_ENGINE_IMAGES, ex.VISUAL_TOKEN_BUDGET)

# What a resumed run must share with the run it continues.
RUN_SETTINGS = {
    'split': SPLIT, 'data_dir': str(DATA_DIR), 'engine': ENGINE, 'temperature': T,
    'pipeline': ex.settings(T), 'text_input_modes': TEXT_INPUT_MODES, 'table_settings': TABLE_SETTINGS,
    'table_representation': TABLE_REPRESENTATION, 'upstream_version': UPSTREAM_VERSION,
    'fusion_modalities': {k: list(v) for k, v in FUSION_MODALITIES.items()},
    'intermediate_orders': INTERMEDIATE_ORDERS, 'late_paradigms': LATE_PARADIGMS,
    'prompt_fingerprint': prompt_fingerprint(),
}

if not DATA_DIR.is_dir():
    raise FileNotFoundError(f'Article directory not found: {DATA_DIR}')
for load, path in ((LOAD_CONTEXTS, CONTEXTS_PATH), (LOAD_LABELS, LABELS_PATH)):
    if load and not Path(path).is_file():
        raise FileNotFoundError(f'Saved input file not found: {path}')
print('Split:', SPLIT, '| contexts:', CONTEXTS_PATH if LOAD_CONTEXTS else 'generate',
      '| labels:', LABELS_PATH if LOAD_LABELS else 'generate')


In [ ]:
if RESUME_STAGE_DIR is None:
    STAGE_DIR = new_run_dir(OUTPUT_ROOT)
    write_json(STAGE_DIR / 'settings.json', RUN_SETTINGS)
else:
    STAGE_DIR = project_path(RESUME_STAGE_DIR)
    saved_settings = read_json(STAGE_DIR / 'settings.json')
    changed = sorted(k for k in set(saved_settings) | set(RUN_SETTINGS) if saved_settings.get(k) != RUN_SETTINGS.get(k))
    if changed:
        raise ValueError(f'Settings differ from the run you want to continue: {changed}. '
                         'Restore them, or start a new run (RESUME_STAGE_DIR = None).')
ex.STAGE_DIR = STAGE_DIR
TIMINGS_PATH = STAGE_DIR / 'timings.json'
timings = read_json(TIMINGS_PATH) if TIMINGS_PATH.is_file() else {}
print('Output directory:', STAGE_DIR)
print('To continue this run, set RESUME_STAGE_DIR = Path(' + repr(str(STAGE_DIR)) + ')')


## Load the model once
Engine limits and vision budgets come from `engine_settings` (up to `ex.MAX_ENGINE_IMAGES` images per request). If vLLM reports that the multimodal profile does not fit `max_model_len`, lower `ex.MAX_ENGINE_IMAGES`.


In [ ]:
from transformers import AutoProcessor
from vllm import LLM

start = time.perf_counter()
# Every stage sends all papers' requests in one batch; vLLM runs up to MAX_NUM_SEQS of them at once
# and queues the rest within GPU memory. Lower it if the KV cache runs out.
MAX_NUM_SEQS = 32
llm = LLM(model=ex.MODEL_NAME, **ENGINE, max_num_seqs=MAX_NUM_SEQS, gpu_memory_utilization=0.9)
processor = AutoProcessor.from_pretrained(ex.MODEL_NAME)
timings['model_load_' + time.strftime('%Y-%m-%d_%H-%M-%S')] = {'seconds': round(time.perf_counter() - start, 3)}


## Load articles
`load_article` keeps every figure and table image and saves an image manifest for later manual labeling. In the extraction stages, only figures (`kind == 'image'`) enter the image modality.


In [ ]:
articles = []
for directory in sorted(DATA_DIR.iterdir()):
    if directory.is_dir():
        source_dir = directory / 'vlm' if (directory / 'vlm').is_dir() else directory
        articles.append(load_article(source_dir))

article_ids = [a['id'] for a in articles]
assert articles, 'No articles found'
assert len(set(article_ids)) == len(article_ids), 'Duplicate article IDs'
write_json(STAGE_DIR / 'image_manifest.json', [item for a in articles for item in a['image_manifest']])
print(len(articles), 'articles;', Counter(c['kind'] for a in articles for c in a['chunks']))


## 1. Shared article context
From this run's folder (when continuing), the saved file (`LOAD_CONTEXTS`), or the model: one request per article with the **entire** converted article and no images (prose, HTML tables, captions/footnotes and figure transcriptions, which stand in for the figures). The frozen `contexts.json` is used by every later stage. `input_sources.json` records where contexts and labels came from.


In [ ]:
contexts_file, sources_file = STAGE_DIR / 'contexts.json', STAGE_DIR / 'input_sources.json'
input_sources = read_json(sources_file) if sources_file.is_file() else {}
if contexts_file.is_file():
    contexts = load_contexts(contexts_file, articles)
elif LOAD_CONTEXTS:
    contexts = load_contexts(CONTEXTS_PATH, articles)  # validated against the articles' source IDs
    input_sources['contexts'] = str(CONTEXTS_PATH)
else:
    start = time.perf_counter()
    contexts = ex.general_pass(llm, processor, T, articles)
    timings['general_pass'] = {'seconds': round(time.perf_counter() - start, 3)}
    input_sources['contexts'] = 'generated'
write_json(contexts_file, contexts)
write_json(sources_file, input_sources)
write_json(TIMINGS_PATH, timings)
print('Contexts:', input_sources['contexts'])


In [ ]:
from copy import deepcopy

CONTEXT_MODE = "empty"

contexts_full = load_contexts(STAGE_DIR / "contexts.json", articles)
contexts = deepcopy(contexts_full)

if CONTEXT_MODE == "empty":
    contexts = {
        article["id"]: {
            "constructs": [],
            "models": [],
            "groups": [],
            "significance_criteria": [],
        }
        for article in articles
    }

elif CONTEXT_MODE != "full":
    raise ValueError(CONTEXT_MODE)

## 2. Classification (once, shared by every variant)
Each source is labelled once (tables with their HTML body); `filtered` and `elementwise` use these labels, `entire` ignores them, and figures are always gated by them. Saved labels are reused as they are; every source without one is classified.


In [ ]:
labels_file = STAGE_DIR / 'labels.json'
if labels_file.is_file():
    labels = read_json(labels_file)
else:
    saved = read_json(LABELS_PATH) if LOAD_LABELS else {}
    labels = {aid: copy.deepcopy(saved.get(aid, {})) for aid in article_ids}
    unlabeled = {aid: [c['id'] for c in a['chunks'] if c['id'] not in labels[aid]] for aid, a in zip(article_ids, articles)}
    start = time.perf_counter()
    labels = ex.classify(llm, processor, T, articles, contexts, labels=labels)
    timings['classify'] = {'seconds': round(time.perf_counter() - start, 3)}
    input_sources['labels'] = dict(path=str(LABELS_PATH) if LOAD_LABELS else None,
                                   classified_now=sum(map(len, unlabeled.values())))
    write_json(labels_file, labels)
    write_json(sources_file, input_sources)
    write_json(TIMINGS_PATH, timings)
missing = {aid: [c['id'] for c in a['chunks'] if c['id'] not in labels.get(aid, {})]
           for aid, a in zip(article_ids, articles)}
assert not any(missing.values()), f'Labels missing for {missing}'
print('Labels:', input_sources['labels'])
print(Counter(label for values in labels.values() for raw in values.values() for label in normalized_labels(raw)))


## 3. Helpers
Each variant is saved as `<variant>.json` (final relations, input for evaluation) in `STAGE_DIR`; its stage files are `<variant>_*.json`. `completed_variants.json` records the exact papers checkpoint, including any `_attemptN` suffix.

Resume first uses the full papers checkpoint. If it was deleted, it recovers the final relations from `<variant>.json`, or the recorded stage's `_after_revision.json`, and loads `_before_revision.json` when available. This recovery needs no model requests and does not recreate deleted papers or request logs. Timings and completion entries are preserved. The overview marks unavailable audit details as unavailable.

A final export can also be reused without a completion entry; in that case the original stage and its before-revision version are unknown. Invalid JSON or mismatched article IDs stop recovery. If a completed variant has no remaining result file, the helper stops with an explanation instead of silently recomputing it. Only variants without a completion entry or a final export run extraction.


In [ ]:
papers, results = {}, {}  # {variant: {aid: paper}} and {variant: {aid: relations}}
COMPLETED_PATH = STAGE_DIR / 'completed_variants.json'
completed = read_json(COMPLETED_PATH) if COMPLETED_PATH.is_file() else {}


def read_saved_relations(path):
    """Validate a relation export before using it as a resumed branch."""
    saved = read_json(path)
    if not isinstance(saved, dict) or set(saved) != set(article_ids):
        raise ValueError(f'Saved results do not match this run\'s article IDs: {path}')
    if any(not isinstance(rows, list) or any(not isinstance(row, dict) for row in rows)
           for rows in saved.values()):
        raise ValueError(f'Expected a list of relations for each article: {path}')
    return saved


def recover_variant(name, checkpoint):
    """Restore relation versions in memory; deleted audits/requests stay unavailable."""
    final_path = STAGE_DIR / f'{name}.json'
    candidates = [final_path]
    prefix = None
    if checkpoint is not None:
        if not checkpoint.name.endswith('_papers.json'):
            raise ValueError(f'Unexpected papers checkpoint name: {checkpoint}')
        prefix = checkpoint.name[:-len('_papers.json')]
        candidates.append(checkpoint.with_name(f'{prefix}_after_revision.json'))
    for source in candidates:
        try:
            after = read_saved_relations(source)
        except FileNotFoundError:
            continue
        break
    else:
        return None

    before = None
    sources = {'after_revision': str(source)}
    if prefix is not None:
        before_path = checkpoint.with_name(f'{prefix}_before_revision.json')
        try:
            before = read_saved_relations(before_path)
        except FileNotFoundError:
            pass
        else:
            sources['before_revision'] = str(before_path)
    if UPSTREAM_VERSION == 'before_revision' and before is None:
        raise FileNotFoundError(
            f'{name}: final relations are available, but the before-revision file is missing '
            'or its original stage is unknown. Restore it to use before_revision inputs.')

    stage_files = str(checkpoint.with_name(f'{prefix}_*.json')) if prefix is not None else str(source)
    restored = {}
    for aid in article_ids:
        paper = dict(relations_after_revision=after[aid], stage_files=stage_files,
                     resume_sources=dict(sources))
        if before is not None:
            paper['relations_before_revision'] = before[aid]
        restored[aid] = paper
    print(f'Reused {name} from {source.name} (full papers/audit details unavailable)')
    return restored


def save_variant(name, fn):
    """Reuse papers or saved relation exports; run only variants with no saved completion."""
    checkpoint = STAGE_DIR / completed[name] if name in completed else None
    saved = None
    if checkpoint is not None:
        try:
            saved = read_json(checkpoint)
        except FileNotFoundError:
            pass
        else:
            print('Reused', name)
    if saved is None:
        saved = recover_variant(name, checkpoint)

    ran = saved is None
    if ran:
        if checkpoint is not None:
            raise FileNotFoundError(
                f'{name} is marked completed, but its papers, final export and after-revision '
                'file are missing. Restore a result file, or deliberately remove this entry '
                'from completed_variants.json to rerun it.')
        start = time.perf_counter()
        saved = fn()
        timings[name] = {'seconds': round(time.perf_counter() - start, 3)}
    papers[name] = saved
    results[name] = ex.relations_of(saved)
    final_path = STAGE_DIR / f'{name}.json'
    if ran or not final_path.is_file():
        write_json(final_path, results[name])
    if ran:
        # Record the exact stage, including _attemptN when applicable.
        stage_files = next(iter(saved.values()))['stage_files']
        completed[name] = Path(stage_files.replace('_*.json', '_papers.json')).name
        write_json(COMPLETED_PATH, completed)
        write_json(TIMINGS_PATH, timings)
    return results[name]


def extract_text(mode, table_setting):
    name = f'text_{mode}_{table_setting}'
    return save_variant(name, lambda: ex.extract(
        llm, processor, T, articles, contexts, 'text', None if mode == 'entire' else labels,
        text_input=mode, text_condition=condition_for(table_setting), stage_name=name))


def run_fusion(paradigm, mode, table_setting):
    # One fusion paradigm on top of text variant text_<mode>_<table_setting>.
    suffix = f'_{mode}_{table_setting}'
    branches = {'text': papers[f'text{suffix}'], 'images': papers['images']}
    if table_setting == 'separate':
        branches['tables'] = papers['tables']
    # Early and late fusion combine the setting's modalities; an intermediate order names its own.
    modalities = FUSION_MODALITIES[table_setting] if paradigm in ('early', 'late_rule', 'late_llm') else None
    return save_variant(paradigm + suffix, lambda: ex.run_paradigm(
        llm, processor, T, articles, contexts, labels, paradigm, text_input=mode, table_setting=table_setting,
        table_representation=TABLE_REPRESENTATION, upstream_version=UPSTREAM_VERSION,
        modalities=modalities, branches=branches, stage_suffix=suffix))


## 4. Text variants
The text is never split: `entire` and `filtered` send one request per article, `elementwise` one per source.


In [ ]:
for table_setting in TABLE_SETTINGS:
    for mode in TEXT_INPUT_MODES:
        extract_text(mode, table_setting)

## 5. Setting B: tables and combination
The table modality runs once (one request per relevant table, always with its body). Each `separate` text variant is then combined with it by rule-based late fusion, followed by an original-source review of text and table bodies. The result `text_<mode>_separate_with_tables` is the counterpart of `text_<mode>_in_text`. The table extraction is also reused by every setting-B fusion variant below.


In [ ]:
if 'separate' in TABLE_SETTINGS:
    save_variant('tables', lambda: ex.extract(
        llm, processor, T, articles, contexts, 'tables', labels,
        table_representation=TABLE_REPRESENTATION, stage_name='tables'))
    for mode in TEXT_INPUT_MODES:
        name = f'text_{mode}_separate_with_tables'
        source = f'text_{mode}_separate'
        upstream = {'text': ex.relations_of(papers[source], UPSTREAM_VERSION),
                    'tables': ex.relations_of(papers['tables'], UPSTREAM_VERSION)}
        save_variant(name, lambda: ex.late_fusion(
            llm, processor, T, articles, contexts, upstream, text_condition='without_html_tables', stage_name=name,
            input_versions={'text': source, 'tables': 'tables', 'version': UPSTREAM_VERSION}))


## 6. Images
- **images**: one request per non-table image classified tested/hypothesized relations; shared by every fusion variant.


In [ ]:
save_variant('images', lambda: ex.run_paradigm(llm, processor, T, articles, contexts, labels, 'images'))


## 7. Early fusion
The variant's text (all of it for `entire`, the relevant sources otherwise) together with every relevant figure. Tables enter as HTML in both settings. With `entire` and `filtered` this is one request per paper; with `elementwise` every source, figures included, is its own request.


In [ ]:
for table_setting in TABLE_SETTINGS:
    for mode in TEXT_INPUT_MODES:
        run_fusion('early', mode, table_setting)


## 8. Intermediate fusion
Starts from the variant's saved text result (or the shared images) and updates it with each later modality; the later modality's update uses the same text mode. Every step is its own stage, e.g. `text_then_tables_filtered_separate` and `text_then_tables_then_image_filtered_separate`.


In [ ]:
for table_setting in TABLE_SETTINGS:
    for mode in TEXT_INPUT_MODES:
        for paradigm in INTERMEDIATE_ORDERS[table_setting]:
            run_fusion(paradigm, mode, table_setting)


## 9. Late fusion
Fuses the variant's text result with the shared images (and tables in B) without rereading sources, then reviews against the original sources. `late_rule` groups deterministically; `late_llm` uses the LLM's merge as answered.


In [ ]:
for table_setting in TABLE_SETTINGS:
    for mode in TEXT_INPUT_MODES:
        for paradigm in LATE_PARADIGMS:
            run_fusion(paradigm, mode, table_setting)
